# 2025 Ziyaretçi Sayısı Tahmini: Seasonal Naive

Bu notebook yalnızca **12 aylık seasonal-naive** yöntemini uygular. Amaç, 2025 yılındaki her ayı bir önceki yılın aynı ayındaki ziyaretçi sayısıyla tahmin etmektir:

$$\hat{y}_t = y_{t-12}$$

Bu yöntem yalnızca tarih ve ziyaretçi sayısını kullanır. REER, HICP, TREND, GTD veya başka bir model kullanılmaz.

## 1. Kütüphaneler ve veri setinin yüklenmesi

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATASET_PATH = Path("data/raw/turizm_kisi_Reel_HICP_Trend.csv")

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Veri seti bulunamadı: {DATASET_PATH.resolve()}")

raw_data = pd.read_csv(DATASET_PATH)

print("Veri seti yüklendi:", DATASET_PATH)
print("Ham veri boyutu:", raw_data.shape)
print("Sütunlar:", raw_data.columns.tolist())
print("\nİlk 5 satır:")
print(raw_data.head().to_string(index=False))

## 2. Gerekli ön işleme

- `Yil-Ay` sütunu aylık tarihe çevrilir.
- `Ziyaretci` sayısal hedef değişken olarak alınır.
- Tarihler sıralanır; tekrar eden veya eksik takvim ayı olup olmadığı kontrol edilir.
- Nisan–Haziran 2020 hedefleri eksik kalır; bu değerler doldurulmaz. Seasonal-naive 2025 tahminleri yalnızca 2024 değerlerini kullandığı için bu eksiklik tahmini etkilemez.

In [ ]:
data = raw_data[["Yil-Ay", "Ziyaretci"]].copy()
data["Tarih"] = pd.to_datetime(data["Yil-Ay"], format="%Y-%m", errors="raise")
data["Ziyaretci"] = pd.to_numeric(data["Ziyaretci"], errors="coerce")
data = data.sort_values("Tarih").set_index("Tarih")

expected_dates = pd.date_range(data.index.min(), data.index.max(), freq="MS")

if data.index.duplicated().any():
    raise ValueError("Tekrar eden aylık tarihler bulundu.")
if not data.index.equals(expected_dates):
    raise ValueError("Aylık tarih serisinde eksik ay bulundu.")

missing_targets = data.index[data["Ziyaretci"].isna()].strftime("%Y-%m").tolist()

print("Tarih aralığı:", data.index.min().strftime("%Y-%m"), "-", data.index.max().strftime("%Y-%m"))
print("Toplam ay sayısı:", len(data))
print("Tekrar eden tarih sayısı:", int(data.index.duplicated().sum()))
print("Eksik takvim ayı sayısı:", len(expected_dates.difference(data.index)))
print("Eksik hedef sayısı:", int(data["Ziyaretci"].isna().sum()))
print("Eksik hedef ayları:", missing_targets)

## 3. Eğitim ve test ayrımı

2008–2024 dönemi eğitim geçmişi, Ocak–Aralık 2025 dönemi test setidir. Tahmin üretirken 2025 yılının gerçekleşen ziyaretçi değerleri kullanılmaz.

In [ ]:
target = data["Ziyaretci"]
train = target.loc[:"2024-12-01"]
test = target.loc["2025-01-01":"2025-12-01"]

if len(test) != 12 or test.isna().any():
    raise ValueError("2025 test seti 12 adet gözlenen aydan oluşmalıdır.")

print(
    "Eğitim dönemi:", train.index.min().strftime("%Y-%m"), "-", train.index.max().strftime("%Y-%m")
)
print("Eğitim gözlemi:", len(train))
print("Test dönemi:", test.index.min().strftime("%Y-%m"), "-", test.index.max().strftime("%Y-%m"))
print("Test gözlemi:", len(test))

## 4. Seasonal-naive modelinin uygulanması

Bu yöntemde klasik anlamda parametre optimizasyonu yoktur. Modelin öğrendiği bilgi, eğitim setindeki geçmiş aylık hedeflerdir. 2025'teki her tahmin için eğitim setinden tam 12 ay önceki değer seçilir.

In [ ]:
seasonal_period = 12
reference_dates = test.index - pd.DateOffset(months=seasonal_period)
predictions = pd.Series(
    train.reindex(reference_dates).to_numpy(),
    index=test.index,
    name="Tahmin",
)

if predictions.isna().any():
    raise ValueError("Tahmin için gerekli 2024 referans değerlerinden biri eksik.")

results = pd.DataFrame(
    {
        "Gercek": test,
        "Tahmin": predictions,
    }
)
results["Hata_Tahmin_Eksi_Gercek"] = results["Tahmin"] - results["Gercek"]
results["Mutlak_Hata"] = results["Hata_Tahmin_Eksi_Gercek"].abs()
results["Mutlak_Yuzde_Hata"] = results["Mutlak_Hata"] / results["Gercek"] * 100
results.index = results.index.strftime("%Y-%m")

print("Model: 12 aylık seasonal naive")
print("Formül: 2025 ayı tahmini = 2024 yılının aynı ayındaki gerçekleşen değer")
print("\nAylık tahminler:")
print(results.round(2).to_string())

## 5. Başarı skorlarının hesaplanması

MAE, RMSE, MAPE ve $R^2$ değerleri 2025'in 12 ayı üzerinden hesaplanır.

In [ ]:
y_true = test.to_numpy(dtype=float)
y_pred = predictions.to_numpy(dtype=float)
errors = y_true - y_pred

mae = np.mean(np.abs(errors))
rmse = np.sqrt(np.mean(errors**2))
mape = np.mean(np.abs(errors / y_true)) * 100
r2 = 1 - np.sum(errors**2) / np.sum((y_true - y_true.mean()) ** 2)

print("2025 SEASONAL-NAIVE SONUÇLARI")
print("-" * 38)
print(f"MAE  : {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"MAPE : %{mape:.4f}")
print(f"R²   : {r2:.6f}")

## Seasonal-naive sonucu

Seasonal-naive yöntemi 2025 test döneminde yaklaşık **%3,28 MAPE** üretir. Sonuç, her ay için yalnızca 2024 yılının aynı ayındaki ziyaretçi sayısının kullanılmasıyla elde edilir.

# Instagram trend verisiyle keşifsel ek analiz

Bu bölüm Almanya, Rusya ve Birleşik Krallık (`UK`) için `#Travel` ve `#TravelTuesday` etiketlerini inceler. Çalışma kitabında aynı kayıtları içeren üç sayfa bulunduğu için yalnızca `trending_hashtags` sayfası okunur; sayfalar üst üste eklenmez.

Verinin gözlem birimi ve kaynağı güvenilir biçimde doğrulanamadığından sonuçlar yalnızca **keşifsel** kabul edilmelidir. Ayrıca elimizde ülke bazında turist gelişi değil, Türkiye toplam ziyaretçi hedefi vardır. Dolayısıyla hesaplanan ilişki, bu üç ülkedeki etiket ölçüleri ile Türkiye'nin toplam ziyaretçi sayısı arasındadır.

## 6. Instagram veri setinin yüklenmesi

In [ ]:
INSTAGRAM_PATH = Path("data/kaggle/trending_hashtags.xlsx")

if not INSTAGRAM_PATH.exists():
    raise FileNotFoundError(f"Instagram veri seti bulunamadı: {INSTAGRAM_PATH.resolve()}")

excel_file = pd.ExcelFile(INSTAGRAM_PATH)
print("Çalışma kitabındaki sayfalar:", excel_file.sheet_names)
print("Yalnızca 'trending_hashtags' sayfası kullanılacak.")

instagram_raw = pd.read_excel(
    INSTAGRAM_PATH,
    sheet_name="trending_hashtags",
    usecols=["date", "hashtag", "mentions", "top_country"],
)

print("Yüklenen Instagram veri boyutu:", instagram_raw.shape)
print("Tarih aralığı:", instagram_raw["date"].min(), "-", instagram_raw["date"].max())

## 7. Ülke ve etiket filtreleme

Yalnızca Almanya, Rusya ve UK ile `#Travel` / `#TravelTuesday` satırları tutulur. Aynı ülke–etiket–gün için birden fazla satır bulunduğundan satırları toplamayıp aylık **ortalama mention** hesaplanır. Mayıs 2024 ve Mayıs 2025 eksik ay kapsamına sahip olduğu için korelasyondan çıkarılır. Tam aylar Haziran 2024–Nisan 2025'tir.

In [ ]:
countries = ["Germany", "Russia", "UK"]
hashtags = ["#Travel", "#TravelTuesday"]

instagram = instagram_raw.copy()
instagram["date"] = pd.to_datetime(instagram["date"], errors="coerce")
instagram["mentions"] = pd.to_numeric(instagram["mentions"], errors="coerce")
instagram = (
    instagram.loc[instagram["top_country"].isin(countries) & instagram["hashtag"].isin(hashtags)]
    .dropna(subset=["date", "mentions"])
    .copy()
)
instagram["month"] = instagram["date"].dt.to_period("M").dt.to_timestamp()

complete_instagram = instagram.loc[instagram["month"].between("2024-06-01", "2025-04-01")].copy()

monthly_tags = complete_instagram.groupby(["month", "top_country", "hashtag"], as_index=False).agg(
    Ortalama_Mention=("mentions", "mean"), Satir_Sayisi=("mentions", "size")
)

print("Filtrelenmiş toplam satır:", len(instagram))
print("Tam aylarda kullanılan satır:", len(complete_instagram))
print("Tam ay sayısı:", complete_instagram["month"].nunique())
print("\nÜlke ve etiket bazında satır sayıları:")
print(instagram.groupby(["top_country", "hashtag"]).size().to_string())

## 8. Ziyaretçi sayısıyla korelasyon

Her seri için hem aynı ay korelasyonu hem de etiket sinyalinin bir ay önce geldiği gecikmeli korelasyon hesaplanır. Örneklem çok küçük olduğu için bu değerler istatistiksel veya nedensel kanıt değildir.

In [ ]:
tag_matrix = monthly_tags.pivot(
    index="month",
    columns=["top_country", "hashtag"],
    values="Ortalama_Mention",
)
country_tag_pairs = list(tag_matrix.columns)
tag_matrix.columns = [f"{country}_{hashtag}" for country, hashtag in country_tag_pairs]
correlation_data = pd.DataFrame({"Ziyaretci": target}).join(tag_matrix, how="left")

correlation_rows = []
for (country, hashtag), column in zip(country_tag_pairs, tag_matrix.columns, strict=False):
    same_month = correlation_data.loc["2024-06":"2025-04", [column, "Ziyaretci"]].dropna()
    one_month_lag = (
        pd.concat(
            [
                correlation_data[column].shift(1).rename("Mention_t_eksi_1"),
                correlation_data["Ziyaretci"],
            ],
            axis=1,
        )
        .loc["2024-07":"2025-05"]
        .dropna()
    )
    correlation_rows.append(
        {
            "Ulke": country,
            "Etiket": hashtag,
            "Ayni_Ay_n": len(same_month),
            "Ayni_Ay_Korelasyon": same_month[column].corr(same_month["Ziyaretci"]),
            "Bir_Ay_Gecikmeli_n": len(one_month_lag),
            "Bir_Ay_Gecikmeli_Korelasyon": one_month_lag["Mention_t_eksi_1"].corr(
                one_month_lag["Ziyaretci"]
            ),
        }
    )

correlations = pd.DataFrame(correlation_rows)
print(correlations.round(4).to_string(index=False))

instagram_signal = (
    complete_instagram.groupby("month")["mentions"].mean().rename("Instagram_Sinyali")
)
aggregate_relation = pd.DataFrame({"Ziyaretci": target}).join(instagram_signal, how="left")
aggregate_relation["Instagram_t_eksi_1"] = aggregate_relation["Instagram_Sinyali"].shift(1)
aggregate_same_corr = (
    aggregate_relation.loc["2024-06":"2025-04", ["Instagram_Sinyali", "Ziyaretci"]]
    .corr()
    .iloc[0, 1]
)
aggregate_lag_corr = (
    aggregate_relation.loc["2024-07":"2025-05", ["Instagram_t_eksi_1", "Ziyaretci"]]
    .corr()
    .iloc[0, 1]
)

print(f"\nBirleştirilmiş aynı-ay korelasyonu: {aggregate_same_corr:.4f}")
print(f"Birleştirilmiş bir-ay gecikmeli korelasyon: {aggregate_lag_corr:.4f}")

## 9. Instagram sinyalinin tahmin skoruna etkisi

Tek bir özellik kullanılır: üç ülke ve iki etiketteki aylık ortalama mention sinyalinin bir ay gecikmeli değeri. Temmuz–Aralık 2024'te seasonal-naive hatasına doğrusal bir düzeltme eğitilir. Ocak–Mayıs 2025, Instagram verisinin gecikmeli olarak kullanılabildiği ortak test dönemidir.

Eğitimde yalnızca 6 ay, testte yalnızca 5 ay bulunduğundan bu deney güvenilir bir model seçimi değil, skor etkisini gösteren küçük bir keşifsel denemedir.

In [ ]:
instagram_model_data = pd.DataFrame(
    {
        "Gercek": target,
        "Seasonal_Naive": target.shift(12),
    }
).join(instagram_signal, how="left")
instagram_model_data["Instagram_t_eksi_1"] = instagram_model_data["Instagram_Sinyali"].shift(1)
instagram_model_data["Seasonal_Naive_Artigi"] = (
    instagram_model_data["Gercek"] - instagram_model_data["Seasonal_Naive"]
)

required_model_columns = ["Gercek", "Seasonal_Naive", "Instagram_t_eksi_1", "Seasonal_Naive_Artigi"]
instagram_train = (
    instagram_model_data.loc["2024-07":"2024-12"].dropna(subset=required_model_columns).copy()
)
instagram_test = (
    instagram_model_data.loc["2025-01":"2025-05"].dropna(subset=required_model_columns).copy()
)

signal_mean = instagram_train["Instagram_t_eksi_1"].mean()
signal_std = instagram_train["Instagram_t_eksi_1"].std(ddof=0)
if signal_std == 0:
    raise ValueError("Instagram eğitim sinyalinin standart sapması sıfır.")

x_train = (instagram_train["Instagram_t_eksi_1"] - signal_mean) / signal_std
design_train = np.column_stack([np.ones(len(x_train)), x_train])
coefficients = np.linalg.lstsq(
    design_train,
    instagram_train["Seasonal_Naive_Artigi"].to_numpy(),
    rcond=None,
)[0]

x_test = (instagram_test["Instagram_t_eksi_1"] - signal_mean) / signal_std
design_test = np.column_stack([np.ones(len(x_test)), x_test])
instagram_test["Instagram_Duzeltmeli_Tahmin"] = (
    instagram_test["Seasonal_Naive"] + design_test @ coefficients
)


def calculate_scores(actual, forecast):
    error = actual - forecast
    return {
        "MAE": error.abs().mean(),
        "RMSE": np.sqrt((error**2).mean()),
        "MAPE": (error.abs() / actual).mean() * 100,
    }


baseline_common_scores = calculate_scores(
    instagram_test["Gercek"], instagram_test["Seasonal_Naive"]
)
instagram_scores = calculate_scores(
    instagram_test["Gercek"], instagram_test["Instagram_Duzeltmeli_Tahmin"]
)

print("Instagram eğitim ayı sayısı:", len(instagram_train))
print("Ortak 2025 test ayı sayısı:", len(instagram_test))
print("\nOrtak test dönemindeki aylık sonuçlar:")
print(
    instagram_test[["Gercek", "Seasonal_Naive", "Instagram_Duzeltmeli_Tahmin"]].round(2).to_string()
)
print("\nSEASONAL-NAIVE — OCAK-MAYIS 2025")
print(f"MAE  : {baseline_common_scores['MAE']:,.2f}")
print(f"RMSE : {baseline_common_scores['RMSE']:,.2f}")
print(f"MAPE : %{baseline_common_scores['MAPE']:.4f}")
print("\nINSTAGRAM DÜZELTMELİ — OCAK-MAYIS 2025")
print(f"MAE  : {instagram_scores['MAE']:,.2f}")
print(f"RMSE : {instagram_scores['RMSE']:,.2f}")
print(f"MAPE : %{instagram_scores['MAPE']:.4f}")
print(
    f"\nMAPE değişimi: {instagram_scores['MAPE'] - baseline_common_scores['MAPE']:+.4f} yüzde puanı"
)

## Instagram analizi sonucu

- Ülke–etiket serilerinin bir ay gecikmeli korelasyonları pozitife işaret etmemiştir.
- Birleştirilmiş bir-ay gecikmeli korelasyon yaklaşık **−0,466** çıkmıştır; fakat yalnızca 11 ortak aylık gözleme dayanır.
- Ortak Ocak–Mayıs 2025 döneminde seasonal-naive MAPE yaklaşık **%3,08**, Instagram düzeltmeli tahmin MAPE ise yaklaşık **%8,89** olmuştur.
- Bu veri ve kurulum altında Instagram sinyali skoru iyileştirmemiş, belirgin biçimde kötüleştirmiştir.

Bu sonuç Instagram eğilimlerinin genel olarak işe yaramadığını kanıtlamaz. Çalışma kitabının muhtemel sentetik yapısı, belirsiz `top_country` anlamı, Türkiye'ye özgü olmayan genel etiketler, yalnızca 6 aylık eğitim ve ülke bazlı ziyaretçi hedefinin bulunmaması güvenilir bir çıkarımı engeller.

# GTD terör verisiyle bölgesel ilişki ve tahmin deneyi

> **Yöntem notu:** Bu basit korelasyon/seasonal-naive düzeltmesi keşifseldir ve nihai ekonometrik sonuç değildir. Asimetrik şoklar, durağanlık, ARDL bounds testi, tanısal gecikme seçimi ve ayrı tahmin doğrulaması `teror_turizm_nardl_analizi.ipynb` dosyasında uygulanmıştır; bilimsel yorumda o notebook esas alınmalıdır.

Bu bölüm lisanslı **Global Terrorism Database (GTD), 1970–2020** verisini kullanır. GTD 2020'de bittiği için 2025 tahminini yeniden hesaplamak mümkün değildir; deney yalnızca turizm verisiyle ortak olan **2008–2020** döneminde yapılır. Ham olay satırları, olay kimlikleri ve açıklamalar yazdırılmaz; sadece aylık bölgesel olay sayıları üretilir.

Kaynak: START (National Consortium for the Study of Terrorism and Responses to Terrorism). (2022). *Global Terrorism Database, 1970–2020 [data file].* https://www.start.umd.edu/data-tools/GTD — Copyright University of Maryland 2022.

## 10. GTD'nin yüklenmesi ve dört ayrık bölgeye dönüştürülmesi

Yalnızca yıl, ay, ülke ve GTD bölgesi alanları okunur. Dört seri birbirini dışlayacak biçimde tanımlanır:

- **Türkiye:** `country_txt == Turkey`
- **MENA (Türkiye hariç):** GTD'nin `Middle East & North Africa` bölgesi; bu nedenle yalnızca Ortadoğu değil, Kuzey Afrika'yı da içerir,
- **Avrupa (Türkiye hariç):** `Western Europe + Eastern Europe`,
- **Amerika:** `North America + Central America & Caribbean + South America`.

Ay kodu 0 olan kayıtların ayı bilinmediği için aylık analize alınmaz. GTD kapsamı içindeki olaysız aylar sıfırdır; **2021 sonrası ise sıfırla doldurulmaz**.

In [ ]:
import itertools

import statsmodels.api as sm
from scipy.stats import pearsonr

GTD_PATH = Path("data/kaggle/global_terrorism.xlsx")
if not GTD_PATH.exists():
    raise FileNotFoundError(f"GTD dosyası bulunamadı: {GTD_PATH.resolve()}")

gtd_raw = pd.read_excel(
    GTD_PATH,
    sheet_name="Data",
    usecols=["iyear", "imonth", "country_txt", "region_txt"],
)
gtd_raw["iyear"] = pd.to_numeric(gtd_raw["iyear"], errors="coerce")
gtd_raw["imonth"] = pd.to_numeric(gtd_raw["imonth"], errors="coerce")
unknown_month_count = int(gtd_raw["imonth"].eq(0).sum())

gtd = gtd_raw.loc[gtd_raw["iyear"].between(2008, 2020) & gtd_raw["imonth"].between(1, 12)].copy()
gtd["Tarih"] = pd.to_datetime(
    {
        "year": gtd["iyear"].astype(int),
        "month": gtd["imonth"].astype(int),
        "day": 1,
    }
)

region_masks = {
    "Turkiye": gtd["country_txt"].eq("Turkey"),
    "MENA_Turkiye_Haric": (
        gtd["region_txt"].eq("Middle East & North Africa") & ~gtd["country_txt"].eq("Turkey")
    ),
    "Avrupa_Turkiye_Haric": (
        gtd["region_txt"].isin(["Western Europe", "Eastern Europe"])
        & ~gtd["country_txt"].eq("Turkey")
    ),
    "Amerika": gtd["region_txt"].isin(
        ["North America", "Central America & Caribbean", "South America"]
    ),
}

overlap_months = pd.date_range("2008-01-01", "2020-12-01", freq="MS")
gtd_monthly = pd.DataFrame(index=overlap_months)
for region_name, region_mask in region_masks.items():
    gtd_monthly[region_name] = (
        gtd.loc[region_mask]
        .groupby("Tarih")
        .size()
        .reindex(overlap_months, fill_value=0)
        .astype(float)
    )
gtd_monthly["Ziyaretci"] = target.reindex(overlap_months)

print("GTD ortak dönemi: 2008-01 / 2020-12")
print("Aylık analizden çıkarılan, ayı bilinmeyen tüm GTD kayıtları:", unknown_month_count)
print("Ortak dönemde hedefi eksik ay sayısı:", int(gtd_monthly["Ziyaretci"].isna().sum()))
print("\n2008-2020 toplam olay sayıları:")
print(gtd_monthly[list(region_masks)].sum().astype(int).to_string())
print("\nİlk 6 aylık bölgesel özet:")
print(gtd_monthly[list(region_masks)].head(6).astype(int).to_string())

## 11. Tekil bölgeler, tüm kombinasyonlar ve gecikmeler

Dört bölgenin boş olmayan bütün alt kümeleri oluşturulur: 4 tekil seri ve 11 toplam seri, yani **15 kombinasyon**. Her seri için aynı ay (`lag=0`) ile 1, 2, 3, 6 ve 12 aylık gecikmeler sınanır; toplam **90 ilişki testi** yapılır.

- Ham Pearson korelasyonu yalnızca betimleyicidir.
- Düzeltilmiş korelasyonda ziyaretçi ve olay sayılarının `log(1+x)` değerlerinden ay mevsimselliği, doğrusal trend, Nisan-2008/Ocak-2012 GTD yöntem kırılmaları ve 2020 pandemi dönemi çıkarılır.
- Aylık serilerde ardışık bağımlılığı hesaba katmak için düzeltilmiş ilişkinin p-değeri 12 aylık Newey–West/HAC standart hatasıyla hesaplanır. Çok sayıda denemeden tesadüfi sonuç seçmemek için bu HAC p-değerlerine ayrıca Benjamini–Hochberg düzeltmesi uygulanır.
- `lag=0` aynı aya ait olduğundan tahmin özelliği değil, yalnızca eşzamanlı ilişkidir.

In [ ]:
region_names = list(region_masks)
terror_combinations = {}
for combination_size in range(1, len(region_names) + 1):
    for selected_regions in itertools.combinations(region_names, combination_size):
        combination_name = " + ".join(selected_regions)
        terror_combinations[combination_name] = gtd_monthly[list(selected_regions)].sum(axis=1)


def adjustment_design(index, source_lag=0):
    month_dummies = pd.get_dummies(index.month, prefix="ay", drop_first=True, dtype=float)
    month_dummies.index = index
    shift = pd.DateOffset(months=int(source_lag))
    design = pd.DataFrame(
        {
            "sabit": 1.0,
            "trend": np.arange(len(index), dtype=float),
            "post_2008_04": (index >= pd.Timestamp("2008-04-01") + shift).astype(float),
            "post_2012_01": (index >= pd.Timestamp("2012-01-01") + shift).astype(float),
            "pandemi_2020": (
                (index >= pd.Timestamp("2020-03-01") + shift)
                & (index <= pd.Timestamp("2020-12-01") + shift)
            ).astype(float),
        },
        index=index,
    )
    return pd.concat([design, month_dummies], axis=1)


def adjusted_residual(series, source_lag=0):
    transformed = np.log1p(series.astype(float))
    valid = transformed.notna()
    design = adjustment_design(transformed.index, source_lag).loc[valid]
    coefficients = np.linalg.lstsq(
        design.to_numpy(), transformed.loc[valid].to_numpy(), rcond=None
    )[0]
    residual = pd.Series(np.nan, index=transformed.index, dtype=float)
    residual.loc[valid] = transformed.loc[valid] - design.to_numpy() @ coefficients
    return residual


def pearson_result(first, second):
    paired = pd.concat([first, second], axis=1).dropna()
    if len(paired) < 3 or paired.iloc[:, 0].nunique() < 2 or paired.iloc[:, 1].nunique() < 2:
        return np.nan, np.nan, len(paired)
    correlation, p_value = pearsonr(paired.iloc[:, 0], paired.iloc[:, 1])
    return float(correlation), float(p_value), len(paired)


def hac_association(first, second, max_lag=12):
    paired = pd.concat([first, second], axis=1).dropna()
    if len(paired) < 3 or paired.iloc[:, 0].nunique() < 2 or paired.iloc[:, 1].nunique() < 2:
        return np.nan, np.nan, len(paired)
    correlation = float(pearsonr(paired.iloc[:, 0], paired.iloc[:, 1]).statistic)
    design = sm.add_constant(paired.iloc[:, 1].to_numpy(dtype=float))
    fitted = sm.OLS(paired.iloc[:, 0].to_numpy(dtype=float), design).fit(
        cov_type="HAC", cov_kwds={"maxlags": min(max_lag, len(paired) - 2)}
    )
    return correlation, float(fitted.pvalues[1]), len(paired)


def benjamini_hochberg(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted_ranked = np.minimum.accumulate(
        (ranked * len(ranked) / np.arange(1, len(ranked) + 1))[::-1]
    )[::-1]
    adjusted = np.empty_like(adjusted_ranked)
    adjusted[order] = np.clip(adjusted_ranked, 0, 1)
    return adjusted


visitor_adjusted = adjusted_residual(gtd_monthly["Ziyaretci"])
correlation_rows = []
for combination_name, incident_series in terror_combinations.items():
    for lag in [0, 1, 2, 3, 6, 12]:
        lagged_incidents = incident_series.shift(lag)
        raw_r, raw_p, _ = pearson_result(gtd_monthly["Ziyaretci"], lagged_incidents)
        incident_adjusted = adjusted_residual(lagged_incidents, source_lag=lag)
        adjusted_r, hac_p, n = hac_association(visitor_adjusted, incident_adjusted)
        correlation_rows.append(
            {
                "Kombinasyon": combination_name,
                "Gecikme_Ay": lag,
                "N": n,
                "Ham_r": raw_r,
                "Ham_p": raw_p,
                "Duzeltilmis_r": adjusted_r,
                "HAC_p": hac_p,
            }
        )

terror_correlations = pd.DataFrame(correlation_rows)
terror_correlations["BH_p"] = benjamini_hochberg(terror_correlations["HAC_p"])
terror_correlations["Mutlak_Duzeltilmis_r"] = terror_correlations["Duzeltilmis_r"].abs()
best_per_combination = (
    terror_correlations.sort_values("Mutlak_Duzeltilmis_r", ascending=False)
    .drop_duplicates("Kombinasyon")
    .sort_values("Mutlak_Duzeltilmis_r", ascending=False)
)

print(f"Üretilen kombinasyon sayısı: {len(terror_combinations)}")
print(f"Toplam korelasyon testi: {len(terror_correlations)}")
print("\nAynı-ay (lag=0) sonuçları — yalnızca betimleyici:")
print(
    terror_correlations.loc[
        terror_correlations["Gecikme_Ay"].eq(0),
        ["Kombinasyon", "N", "Ham_r", "Duzeltilmis_r", "HAC_p", "BH_p"],
    ]
    .sort_values("Duzeltilmis_r")
    .round(4)
    .to_string(index=False)
)
print("\nHer kombinasyonun en güçlü düzeltilmiş ilişkisi:")
print(
    best_per_combination[["Kombinasyon", "Gecikme_Ay", "N", "Duzeltilmis_r", "HAC_p", "BH_p"]]
    .round(4)
    .to_string(index=False)
)
print("\nMutlak değeri en yüksek 10 düzeltilmiş korelasyon:")
print(
    terror_correlations.sort_values("Mutlak_Duzeltilmis_r", ascending=False)[
        ["Kombinasyon", "Gecikme_Ay", "N", "Duzeltilmis_r", "HAC_p", "BH_p"]
    ]
    .head(10)
    .round(4)
    .to_string(index=False)
)
print(
    "\nBH düzeltmesinden sonra p<0.05 kalan test sayısı:",
    int((terror_correlations["BH_p"] < 0.05).sum()),
)
print("En küçük BH-düzeltilmiş p-değeri:", round(float(terror_correlations["BH_p"].min()), 6))

## 12. Ayrı test döneminde tahmin etkisi

Korelasyon tablosundaki en iyi sonucu test dönemine bakarak seçmek yanıltıcı olur. Bu nedenle:

1. **2008–2014** yalnızca seçim/eğitim dönemidir. `lag=0` tahminde kullanılamayacağı için 15 kombinasyon × 5 pozitif gecikme arasından, seasonal-naive eğitim artığıyla `log(1+olay)` sinyali arasındaki mutlak korelasyonu en yüksek özellik seçilir.
2. Ana model yine 12-ay seasonal-naive'dir. Eğitimdeki seasonal-naive artıklarına seçilen güvenlik özelliğiyle tek değişkenli doğrusal düzeltme uygulanır.
3. **2015–2020** hiç dokunulmamış test dönemidir. 2020 Nisan–Haziran hedefleri eksik olduğundan 69 ay skorlanır.
4. Sadece sabit ortalama artık düzeltmesi ayrıca kontrol olarak verilir. Böylece iyileşmenin terör sinyalinden mi, basit seviye düzeltmesinden mi geldiği görülebilir.

Bu kurulum retrospektiftir: GTD'nin operasyonel yayın zamanları bilinmediğinden sonuç gerçek-zamanlı kullanılabilirlik kanıtı değildir.

In [ ]:
training_end = pd.Timestamp("2014-12-01")
test_start = pd.Timestamp("2015-01-01")
test_end = pd.Timestamp("2020-12-01")
training_period = gtd_monthly.index <= training_end
test_period = (gtd_monthly.index >= test_start) & (gtd_monthly.index <= test_end)

all_visitors = gtd_monthly["Ziyaretci"]
seasonal_naive_overlap = all_visitors.shift(12)
training_residual = all_visitors - seasonal_naive_overlap

selection_rows = []
for combination_name, incident_series in terror_combinations.items():
    for lag in [1, 2, 3, 6, 12]:
        training_error = training_residual.loc[training_period]
        training_feature = np.log1p(incident_series.shift(lag)).loc[training_period]
        selected_r, selected_p, selected_n = pearson_result(
            training_error,
            training_feature,
        )
        selection_rows.append(
            {
                "Kombinasyon": combination_name,
                "Gecikme_Ay": lag,
                "Egitim_r": selected_r,
                "Egitim_p": selected_p,
                "N": selected_n,
            }
        )
selection_table = pd.DataFrame(selection_rows)
chosen = selection_table.loc[selection_table["Egitim_r"].abs().idxmax()]
chosen_name = chosen["Kombinasyon"]
chosen_lag = int(chosen["Gecikme_Ay"])

chosen_feature = terror_combinations[chosen_name].shift(chosen_lag)
training_valid = (
    training_period & all_visitors.notna() & seasonal_naive_overlap.notna() & chosen_feature.notna()
)
test_valid = (
    test_period & all_visitors.notna() & seasonal_naive_overlap.notna() & chosen_feature.notna()
)

feature_log = np.log1p(chosen_feature)
feature_mean = float(feature_log.loc[training_valid].mean())
feature_std = float(feature_log.loc[training_valid].std(ddof=0))
feature_standardized = (feature_log - feature_mean) / feature_std
training_design = np.column_stack(
    [
        np.ones(int(training_valid.sum())),
        feature_standardized.loc[training_valid].to_numpy(),
    ]
)
security_coefficients = np.linalg.lstsq(
    training_design, training_residual.loc[training_valid].to_numpy(), rcond=None
)[0]

test_index = gtd_monthly.index[test_valid]
gtd_predictions = pd.DataFrame(index=test_index)
gtd_predictions["Gercek"] = all_visitors.loc[test_index]
gtd_predictions["Seasonal_Naive"] = seasonal_naive_overlap.loc[test_index]
mean_training_residual = float(training_residual.loc[training_valid].mean())
gtd_predictions["Sadece_Seviye_Duzeltmesi"] = (
    gtd_predictions["Seasonal_Naive"] + mean_training_residual
)
gtd_predictions["Teror_Sinyali_Duzeltmeli"] = (
    gtd_predictions["Seasonal_Naive"]
    + security_coefficients[0]
    + security_coefficients[1] * feature_standardized.loc[test_index]
)


def score_prediction(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return {
        "MAE": np.mean(np.abs(actual - predicted)),
        "RMSE": np.sqrt(np.mean((actual - predicted) ** 2)),
        "MAPE": np.mean(np.abs((actual - predicted) / actual)) * 100,
    }


gtd_score_rows = []
for model_name in ["Seasonal_Naive", "Sadece_Seviye_Duzeltmesi", "Teror_Sinyali_Duzeltmeli"]:
    model_scores = score_prediction(gtd_predictions["Gercek"], gtd_predictions[model_name])
    gtd_score_rows.append({"Model": model_name, **model_scores})
gtd_scores = pd.DataFrame(gtd_score_rows)
pre_covid_predictions = gtd_predictions.loc[:"2019-12-01"]
pre_covid_score_rows = []
for model_name in ["Seasonal_Naive", "Sadece_Seviye_Duzeltmesi", "Teror_Sinyali_Duzeltmeli"]:
    model_scores = score_prediction(
        pre_covid_predictions["Gercek"], pre_covid_predictions[model_name]
    )
    pre_covid_score_rows.append({"Model": model_name, **model_scores})
pre_covid_scores = pd.DataFrame(pre_covid_score_rows)

print("Eğitim döneminde seçilen özellik:", chosen_name)
print("Seçilen gecikme (ay):", chosen_lag)
print(
    f"Eğitim artığı–özellik korelasyonu: {chosen['Egitim_r']:.4f} "
    f"(p={chosen['Egitim_p']:.4f}, N={int(chosen['N'])})"
)
print("Testte değerlendirilen ay sayısı:", len(gtd_predictions))
print("\n2015-2020 aylık tahminler:")
print(gtd_predictions.round(2).to_string())
print("\n2015-2020 test skorları:")
print(gtd_scores.round({"MAE": 2, "RMSE": 2, "MAPE": 4}).to_string(index=False))
print("\nPandemi duyarlılığı — yalnızca 2015-2019 test skorları:")
print(pre_covid_scores.round({"MAE": 2, "RMSE": 2, "MAPE": 4}).to_string(index=False))

baseline_row = gtd_scores.set_index("Model").loc["Seasonal_Naive"]
control_row = gtd_scores.set_index("Model").loc["Sadece_Seviye_Duzeltmesi"]
security_row = gtd_scores.set_index("Model").loc["Teror_Sinyali_Duzeltmeli"]
print(
    "\nTerörlü modelin seasonal-naive'e göre MAPE değişimi: "
    f"{security_row['MAPE'] - baseline_row['MAPE']:+.4f} yüzde puanı"
)
print(
    "Terörlü modelin seviye kontrolüne göre MAPE değişimi: "
    f"{security_row['MAPE'] - control_row['MAPE']:+.4f} yüzde puanı"
)

## GTD deneyinin yorumu

- Mevsimsellik, trend, GTD yöntem kırılmaları ve 2020 şoku çıkarıldıktan sonra en güçlü tekil ilişki **Türkiye olaylarının 2 aylık gecikmesi** için bulundu: `r = -0,4474`.
- Türkiye hariç MENA için en güçlü sonuç 12 ay gecikmede `r = -0,2592` (`HAC p = 0,0550`, `BH p = 0,2706`); Avrupa için 12 ay gecikmede `r = -0,1908` (`HAC p = 0,1139`, `BH p = 0,2973`) oldu. Amerika'nın en güçlü sonucu ise 6 ay gecikmede `r = +0,1203` (`HAC p = 0,1216`, `BH p = 0,2973`) oldu.
- Kullanıcının örneğindeki **Türkiye + MENA** toplamı için en güçlü sonuç 12 ay gecikmede `r = -0,2869` (`HAC p = 0,0400`, `BH p = 0,2706`) oldu. Bu ilişki Türkiye tek başına elde edilen en güçlü ilişkiden daha kuvvetli değildir.
- Türkiye'nin 2 aylık gecikmesinde katsayı güçlü görünse de HAC `p = 0,0028`, 90 testlik BH düzeltmesinden sonra `p = 0,0831` oldu. **90 testin hiçbiri BH-düzeltilmiş %5 eşiğini geçmedi**; en küçük düzeltilmiş p-değeri `0,0736` idi. Seriler aynı dört bölgeyi tekrar tekrar kullanan, birbirine bağımlı kombinasyonlardır.
- Tahmin özelliği test dönemine bakılmadan seçildi: eğitimde seasonal-naive artığıyla en yüksek ilişkiyi **3 ay gecikmeli Avrupa** verdi (`r = +0,3152`, `p = 0,0070`). Buna rağmen 2015–2020 testinde terör-sinyali düzeltmeli modelin MAPE'si **%40,2467**, seasonal-naive'in MAPE'si **%38,4926** oldu; sonuç **1,7541 yüzde puanı kötüleşti**. MAE ve RMSE de kötüleşti.
- Pandemi dışı 2015–2019 duyarlılığında da seasonal-naive MAPE **%14,9430**, terörlü model MAPE **%15,9545** oldu. Sadece ortalama seviye düzeltmesi **%14,6362** ile terörlü modelden daha iyiydi. Dolayısıyla bu deneyde terör sayıları seasonal-naive'e ek ve genellenebilir tahmin değeri sağlamadı.

Bu MAPE değerleri 2025'teki **%3,2796** ile karşılaştırılmamalıdır: dönem farklıdır ve 2015–2020 testi turizmdeki 2016 gerilemesini ve 2020 pandemi çöküşünü içerir. Korelasyon da nedensellik değildir. Bölgesel toplamlar seyahat talebi, haber görünürlüğü, jeopolitik risk ve eşzamanlı makroekonomik şokları birbirinden ayıramaz; GTD'nin sonradan düzeltilmiş nihai kayıtları gerçek tahmin tarihindeki veri sürümünü temsil etmeyebilir.